# Day 24 — Batch Prediction API

**Day 23 results from your uploaded notebook:** both manifests were found in their existing folders; all three ML model artifact paths exist and each model has 15 ordered features; Django system check passed; all six Day 21 pagination tests passed.

**Still pending:** merge the Day 23 URL patch and test those endpoints over HTTP. This notebook adds a batch endpoint using the Day 23 ML model configuration. It does not enable LSTM/GRU inference yet.

In [1]:
from pathlib import Path
import sys, subprocess, shutil
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    roots = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if roots:
        REPO_ROOT = roots[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
DAY23_VIEW = APP_DIR / "prediction_views.py"
assert MANAGE_PY.exists(), f"Missing {MANAGE_PY}"
assert DAY23_VIEW.exists(), f"Run Day 23 first; missing {DAY23_VIEW}"
print("Repository:", REPO_ROOT)
print("Day 23 module:", DAY23_VIEW)


Repository: D:\Git\Personal\ipl-player-performance
Day 23 module: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\prediction_views.py


In [2]:
module_source = 'from rest_framework import status\nfrom rest_framework.response import Response\nfrom rest_framework.views import APIView\nfrom .prediction_views import MODEL_CONFIGS, _predict\n\nclass BatchPredictionAPIView(APIView):\n    MAX_BATCH_SIZE = 500\n\n    def post(self, request):\n        target = request.data.get("target")\n        records = request.data.get("records")\n        config = next((c for c in MODEL_CONFIGS if c["target"] == target), None)\n        if config is None:\n            return Response(\n                {"detail": "Unsupported target.", "supported_targets": [c["target"] for c in MODEL_CONFIGS]},\n                status=status.HTTP_400_BAD_REQUEST,\n            )\n        if not isinstance(records, list) or not records:\n            return Response({"detail": "\'records\' must be a non-empty list."}, status=status.HTTP_400_BAD_REQUEST)\n        if len(records) > self.MAX_BATCH_SIZE:\n            return Response({"detail": f"Batch exceeds maximum of {self.MAX_BATCH_SIZE} records."},\n                            status=status.HTTP_400_BAD_REQUEST)\n        results = []\n        for index, record in enumerate(records):\n            try:\n                value, error = _predict(config, record)\n                if error:\n                    return Response({"record_index": index, "error": error}, status=status.HTTP_400_BAD_REQUEST)\n                results.append({"record_index": index, "target": target, "prediction": value})\n            except FileNotFoundError as exc:\n                return Response({"detail": str(exc)}, status=status.HTTP_503_SERVICE_UNAVAILABLE)\n            except Exception:\n                return Response({"detail": "Batch prediction failed. Check server logs for the underlying error."},\n                                status=status.HTTP_500_INTERNAL_SERVER_ERROR)\n        return Response({"target": target, "count": len(results), "predictions": results})\n'
compile(module_source, "batch_prediction_views.py", "exec")
module_path = APP_DIR / "batch_prediction_views.py"
if module_path.exists():
    shutil.copy2(module_path, module_path.with_suffix(".py.day24_backup"))
module_path.write_text(module_source, encoding="utf-8")
print("Generated:", module_path)


Generated: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\batch_prediction_views.py


In [3]:
url_patch = '''# Day 24 batch prediction route — merge into existing django_api/ipl_api/urls.py
from .batch_prediction_views import BatchPredictionAPIView

# Add inside existing urlpatterns:
path("predict/batch/", BatchPredictionAPIView.as_view(), name="predict-batch"),
'''
patch_path = APP_DIR / "day24_batch_prediction_urls_patch.txt"
patch_path.write_text(url_patch, encoding="utf-8")
print("Created:", patch_path)
print("Preserve existing urls.py and merge both Day 23 and Day 24 route patches.")


Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\day24_batch_prediction_urls_patch.txt
Preserve existing urls.py and merge both Day 23 and Day 24 route patches.


In [4]:
compile(module_path.read_text(encoding="utf-8"), str(module_path), "exec")
print("Batch view syntax: OK")

def run_cmd(args):
    p = subprocess.run(args, cwd=str(DJANGO_ROOT), text=True, capture_output=True)
    print("$", " ".join(map(str, args)))
    print("Return code:", p.returncode)
    print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p

django_check = run_cmd([sys.executable, str(MANAGE_PY), "check"])
day21_tests = run_cmd([sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "1"])


Batch view syntax: OK
$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py check
Return code: 0
System check identified no issues (0 silenced).

$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py test ipl_api.tests.test_pagination_utils_day21 --verbosity 1
Return code: 0
Found 6 test(s).
System check identified no issues (0 silenced).

......
----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK



In [5]:
report = f'''# Day 24 — Batch Prediction API

Generated: {datetime.now().isoformat(timespec="seconds")}

## Implemented
- `django_api/ipl_api/batch_prediction_views.py`
- `django_api/ipl_api/day24_batch_prediction_urls_patch.txt`
- Batch request format: `{{"target": "next_match_runs", "records": [{{"feature_1": 1, "...": 2}}]}}`
- Preserves record order; returns record indexes.
- Rejects unsupported targets, empty/non-list records, and batches above 500.
- Reuses Day 23 model configuration and feature validation.

## Checks
- Python syntax: passed.
- Django check return code: {django_check.returncode}.
- Day 21 test return code: {day21_tests.returncode}.

## Still required
Merge the URL patch, confirm Day 23 routes are registered, then test valid/invalid payloads over HTTP. These system checks do not prove that the batch endpoint is registered or that model inference succeeds.
'''
(REPO_ROOT / "docs").mkdir(exist_ok=True)
(REPO_ROOT / "docs" / "day24_batch_prediction_api.md").write_text(report, encoding="utf-8")
print("Saved docs/day24_batch_prediction_api.md")


Saved docs/day24_batch_prediction_api.md


## Completion checklist

- [ ] Run the notebook from the repository root in the `ipl-player-performance` environment.
- [ ] Merge the URL patch into the existing `urls.py`.
- [ ] Run `python manage.py check`.
- [ ] Test a batch of two valid records for each target.
- [ ] Test missing features, non-numeric values, unsupported targets, empty records, and over-limit batches.
- [ ] Compare API results against direct model predictions for identical feature vectors.

Do not mark Day 24 complete until the HTTP smoke tests pass.